In [1]:
import sys

sys.path.append("/mnt/d/work/smolagents/src/")

In [2]:
import importlib
import os
from dotenv import load_dotenv
from smolagents import (
    ToolCallingAgent,
    Tool,
    ChatMessage,
    PromptTemplates,
    ActionStep,
    AgentGenerationError,
    LogLevel,
    AgentParsingError,
    HfApiModel,
    DuckDuckGoSearchTool,
    ToolCall,
    Panel,
    Text,
    AgentAudio,
    AgentImage,
    YELLOW_HEX,
    OpenAIServerModel,
    tool
)
from typing import List, Dict, Optional, Callable, Union, Any
import yaml
import json
from uuid import uuid4
import requests

In [3]:
load_dotenv()

True

In [4]:
with open("/mnt/d/work/smolagents/src/smolagents/prompts/tool_calling_agent2.yaml","rb") as file:
    prompt_templates=yaml.safe_load(file)

In [5]:
@tool
def get_weather(location: str) -> dict:
    """this function takes the location and returns the weather statistics:
    Args:
        location: place for which we want the weather statistics like temperature, weather description,wind speed, wind degree, wind direction,pressure
                  precipitation,humidity,cloud cover ,feels like ,uv index and visibility.
    """
    access_key = os.getenv("WEATHER_API_KEY")
    endpoint = f"http://api.weatherstack.com/current?access_key={access_key}&query={location}"
    resp = requests.get(endpoint)
    return resp.json()


In [6]:
class ReActToolCallingAgent(ToolCallingAgent):
    """
    This agent uses JSON-like tool calls, using method `model.get_tool_call` to leverage the LLM engine's tool calling capabilities.

    Args:
        tools (`list[Tool]`): [`Tool`]s that the agent can use.
        model (`Callable[[list[dict[str, str]]], ChatMessage]`): Model that will generate the agent's actions.
        prompt_templates ([`~agents.PromptTemplates`], *optional*): Prompt templates.
        planning_interval (`int`, *optional*): Interval at which the agent will run a planning step.
        **kwargs: Additional keyword arguments.
    """
    def step(self, memory_step: ActionStep) -> Union[None, Any]:
        memory_messages = self.write_memory_to_messages()
        self.input_messages = memory_messages
        try:
            model_message: ChatMessage = self.model(
                memory_messages,
                tools_to_call_from=None,
                stop_sequences=["Observation:", "Calling tools:"],
            )
            memory_step.model_output_message = model_message
            # print(model_message)
        except Exception as e:
            raise AgentGenerationError(
                f"Error in generating tool call with model:\n{e}", self.logger
            ) from e
        self.logger.log_markdown(
            content=(
                model_message.content
                if model_message.content
                else str(model_message.raw)
            ),
            title="Output message of the LLM:",
            level=LogLevel.DEBUG,
        )
        # print(model_message.content)
        thought, action = model_message.content.split("Action:")
        self.logger.log_markdown(content=(thought), title="Thought")
        if len(action) == 0:
            raise AgentParsingError(
                "Model did not call any tools. Call `final_answer` tool to return a final answer.",
                self.logger,
            )
        tool_call = json.loads(action)
        tool_name, tool_call_id = tool_call['name'], str(uuid4())
        tool_arguments = tool_call['arguments']
        memory_step.tool_calls = [
            ToolCall(name=tool_name, arguments=tool_arguments, id=tool_call_id)
        ]
        # Execute
        self.logger.log(
            Panel(
                Text(f"Calling tool: '{tool_name}' with arguments: {tool_arguments}")
            ),
            level=LogLevel.INFO,
        )
        if tool_name == "final_answer":
            if isinstance(tool_arguments, dict):
                if "answer" in tool_arguments:
                    answer = tool_arguments["answer"]
                else:
                    answer = tool_arguments
            else:
                answer = tool_arguments
            if (
                isinstance(answer, str) and answer in self.state.keys()
            ):  # if the answer is a state variable, return the value
                final_answer = self.state[answer]
                self.logger.log(
                    f"[bold {YELLOW_HEX}]Final answer:[/bold {YELLOW_HEX}] Extracting key '{answer}' from state to return value '{final_answer}'.",
                    level=LogLevel.INFO,
                )
            else:
                final_answer = answer
                self.logger.log(
                    Text(f"Final answer: {final_answer}", style=f"bold {YELLOW_HEX}"),
                    level=LogLevel.INFO,
                )

            memory_step.action_output = final_answer
            return final_answer
        else:
            if tool_arguments is None:
                tool_arguments = {}
            observation = self.execute_tool_call(tool_name, tool_arguments)
            observation_type = type(observation)
            if observation_type in [AgentImage, AgentAudio]:
                if observation_type == AgentImage:
                    observation_name = "image.png"
                elif observation_type == AgentAudio:
                    observation_name = "audio.mp3"
                # TODO: observation naming could allow for different names of same type

                self.state[observation_name] = observation
                updated_information = f"Stored '{observation_name}' in memory."
            else:
                updated_information = str(observation).strip()
            self.logger.log(
                f"Observations: {updated_information.replace('[', '|')}",  # escape potential rich-tag-like components
                level=LogLevel.INFO,
            )
            memory_step.observations = updated_information
            return None

In [7]:
hf_model=HfApiModel(token=os.getenv("HF_TOKEN"))
opeai_model=OpenAIServerModel(model_id="gpt-4")

In [8]:
agent=ReActToolCallingAgent(tools=[DuckDuckGoSearchTool(max_results=3),get_weather],model=opeai_model,prompt_templates=prompt_templates)

In [11]:
agent.run(task="who is the president of India?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ who is the president of India?                                                                                  │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4 ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Thought ───────────────────────────────────────────────────────────────────────────────────────────────────────────
Thought: To get the current president of India, we can use the web_search tool with a specific query.              
                                                                                                                   
                                                                                                                   

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'Current president of India'}                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: ## Search Results

|List of presidents of India - Wikipedia](https://en.wikipedia.org/wiki/List_of_presidents_of_India)
The current president of India is Droupadi Murmu, who took office on 25 July 2022. She is the second woman and the 
first tribal person to hold the office. See the full list of 15 presidents of India since 1950, their terms, 
parties, and oaths.

|Droupadi Murmu - Wikipedia](https://en.wikipedia.org/wiki/Droupadi_Murmu)
Droupadi Murmu (Hindi: |d̪ɾɔːpəd̪iː mʊɾmuː], born Durgi Biranchi Tudu, Hindi: |d̪ʊɾɡiː bɪɾəɲtʃiː t̪ʊd̪uː]; born 20 June
1958) is an Indian politician who has been serving as the president of India since 2022. She won the 2022 
presidential election as the Bharatiya Janata Party (BJP) candidate. |2] She is the first person belonging to a 
tribal community and also the ...

|Home | President of India](https://www.presidentofindia.gov.in/)
The President of India, Smt Droupadi Murmu graced the closing ceremony of the commemoration of 90th year of the 
Reserve Bank of India in Mumbai on April 1, 2025. The President of India, Smt Droupadi Murmu inaugurated a two-day 
National Conference on 'Environment - 2025' in New Delhi on March 29, 2025.

...

Thought ───────────────────────────────────────────────────────────────────────────────────────────────────────────
Thought: From the result of the web search, we can see that the current president of India is Droupadi Murmu.      
                                                                                                                   
                                                                                                                   

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The current president of India is Droupadi Murmu.'}    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final answer: The current president of India is Droupadi Murmu.

[Step 2: Duration 350.73 seconds| Input tokens: 3,036 | Output tokens: 103]

'The current president of India is Droupadi Murmu.'